In [1]:
import torch
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from collections import Counter

In [2]:
def ret_type(codes, sep = ":"):
    """
    codes should be a list []
    """
    codes_split = [code.split(sep)[0] for code in codes]
    return(codes_split)

In [3]:
def get_inst_index(index_type, MGB_row):
    # return the index_type in a inst.
    index_type_MGB = []
    for index_type1 in index_type:
        index_type1_MGB = np.intersect1d(index_type1, MGB_row)
        index_type_MGB.append(index_type1_MGB)
    return index_type_MGB    

In [4]:
def find_desc(name, name_desc):
    index = np.concatenate([np.where(name_desc.iloc[0:6917,0] == x)[0] for x in name])
    desc = name_desc.iloc[index, 1]
    return desc

In [5]:
def get_int_uni_pair(one_pair, two_pair, NOT=False):
    '''
    get unique pairs in intersection(or not)
    '''
    one_pair_set = set(tuple(sorted(pair)) for pair in one_pair)
    two_pair_set = set(tuple(sorted(pair)) for pair in two_pair)
    intersection = one_pair_set.intersection(two_pair_set)
    intersection = [tuple(sorted(pair)) for pair in intersection]
    if NOT is False:
        return np.unique(intersection,axis=0)
    else:
        not_in = []
        not_in.extend([pair for pair in one_pair if tuple(sorted(pair)) not in intersection])
        not_in.extend([pair for pair in two_pair if tuple(sorted(pair)) not in intersection])
        not_in = [tuple(sorted(pair)) for pair in not_in]
        return np.unique(not_in,axis=0)


In [22]:
def get_union_setdiff(one_pair, two_pair, three_pair, common_one_pair):
    one_pair_set = set(tuple(sorted(pair)) for pair in one_pair)
    two_pair_set = set(tuple(sorted(pair)) for pair in two_pair)
    three_pair_set = set(tuple(sorted(pair)) for pair in three_pair)
    union = one_pair_set.union(two_pair_set)
    union = union.union(three_pair_set)
    diff_union = union.symmetric_difference(common_one_pair)
    diff_union = [tuple(sorted(pair)) for pair in diff_union]
    return np.unique(diff_union, axis=0)

In [33]:
def get_common_code(name_desc, MGB_row, VA_row, UP_row, type_need = ['CCS','PheCode','RXNORM','LOINC']):
    type_code = ret_type(name_desc.iloc[:,0])
    Leaf_code = name_desc.iloc[np.where([s.startswith("LOINC:") and not s.startswith("LOINC:LP") for s in name_desc.iloc[:,0] ])[0],0]
    common_code = np.intersect1d(np.intersect1d(name_desc.iloc[MGB_row,0], name_desc.iloc[VA_row,0]), name_desc.iloc[UP_row,0])
    common_code = np.unique(np.setdiff1d(common_code, Leaf_code))
    common_type = ret_type(common_code)
    # print(Counter(common_type))
    
    all_type_code = []
    for type_ in type_need:
        this_type_code = common_code[np.where(np.array(common_type) == type_)[0]]
        all_type_code.append(this_type_code)
    return all_type_code
        

def get_dataframe(common_pair, name_desc):
    common_pair = np.array(common_pair)
    desc1 = find_desc(common_pair[:,0], name_desc)
    desc2 = find_desc(common_pair[:,1], name_desc)
    common_df = np.column_stack([common_pair[:,0], common_pair[:,1], desc1, desc2])
    common_df = pd.DataFrame(common_df)
    common_df.columns = ['name1', 'name2', 'desc1', 'desc2']
    return common_df

def get_sppmi_edge(cos_sim, name_list1, name_list2,  quantile=0.01, shift=0):
    cos_sim = torch.tensor(cos_sim)
    num_rows, num_cols = cos_sim.shape
    # do not consider the same code pair
    if num_rows == num_cols:
        mask = torch.eye(num_rows).bool()
        cos_sim[mask] = float('-inf')
    cos_sim_1d = cos_sim.view(-1)

    threshold = torch.quantile(cos_sim_1d, 1 - quantile)
    topk_indices = torch.where(cos_sim_1d >= threshold)[0]
    topk_values = cos_sim_1d[topk_indices]
    # plt.hist(topk_values, alpha=0.3)
    
    row_indices = topk_indices // num_cols + shift
    col_indices = topk_indices % num_cols + shift
    
    row_names = name_list1[row_indices.numpy()]
    col_names = name_list2[col_indices.numpy()]
    
    highest_names = list(zip(row_names.tolist(), col_names.tolist()))
    return highest_names, topk_values

def get_one_type_edge(sppmi_emb, name_desc, MGB_row, VA_row, UP_row, type_1, type_2):
    # get common code names from three inst.
    common_code =  get_common_code(name_desc, MGB_row, VA_row, UP_row, type_need = [type_1, type_2])
    common_type1 = common_code[0]
    common_type2 = common_code[1]
    # get index from three inst. index_type1 size N*3
    index_type1 = np.array([np.where(name_desc.iloc[:,0] == x)[0] for x in common_type1]) 
    index_type2 = np.array([np.where(name_desc.iloc[:,0] == x)[0] for x in common_type2])
    # range(3) is MGB, VA, UPMC inst.
    all_pair = []
    for i in range(3):
        inst_index_type1 = index_type1[:,i]
        inst_index_type2 = index_type2[:,i]
        name_list1 = name_desc.iloc[inst_index_type1,0]
        name_list2 = name_desc.iloc[inst_index_type2,0]
        cos_sim = np.matmul(np.array(sppmi_emb.iloc[inst_index_type1,:]), np.array(sppmi_emb.iloc[inst_index_type2,:]).transpose())
        get_inst_pair = get_sppmi_edge(cos_sim, name_list1.values, name_list2.values, quantile=0.01)
        all_pair.append(get_inst_pair)
    return all_pair

def get_all_type_edge(sppmi_emb, name_desc, MGB_row, VA_row, UP_row, type_all = ['CCS','PheCode','RXNORM','LOINC']):
    # all_pair is a length=10 pairs[10 types]
    # every type is a length 3 list[3 inst.]
    # every inst is a length 2 list [pair + cos sim]
    all_pair = []
    for i in range(len(type_all)):
        for j in range(i, len(type_all)):
            one_pair = get_one_type_edge(sppmi_emb, name_desc, MGB_row, VA_row, UP_row, type_all[i], type_all[j])
            all_pair.append(one_pair)
    return all_pair

def get_type_pair(type_all):
    pairs = []
    for i in range(len(type_all)):
        for j in range(i, len(type_all)):
            pair = type_all[i] + '&' + type_all[j]
            pairs.append(pair)
    return pairs

def get_final_pairs(sppmi_emb, name_desc, MGB_row, VA_row, UP_row, type_all = ['CCS','PheCode','RXNORM','LOINC']):
    
    all_pair = get_all_type_edge(sppmi_emb, name_desc, MGB_row, VA_row, UP_row, type_all = ['CCS','PheCode','RXNORM','LOINC'])
    type_pair = get_type_pair(type_all)
    common_pair = []
    No_com_pair = []
    
    for i in range(len(all_pair)):
        print(type_pair[i])
        one_pair = all_pair[i]
        common_one_pair = get_int_uni_pair(get_int_uni_pair(one_pair[0][0], one_pair[1][0]), one_pair[2][0])
        # No_com_one_pair = get_int_uni_pair(get_int_uni_pair(one_pair[0][0], one_pair[1][0], NOT=True), one_pair[2][0], NOT=True)
        No_com_one_pair = get_union_setdiff(one_pair[0][0], one_pair[1][0], one_pair[2][0], [tuple(x) for x in common_one_pair])
        print(f'{len(common_one_pair)} common pairs and {len(No_com_one_pair)} uncommon pairs.')
        common_pair.extend(common_one_pair)
        No_com_pair.extend(No_com_one_pair)
    return common_pair, No_com_pair

In [6]:
sppmi_emb = pd.read_csv('https://han-attention.s3.amazonaws.com/input/emb/mat_svd_self_rotate.csv')

name_desc = pd.read_csv('https://han-attention.s3.amazonaws.com/input/name_desc/name_desc_all_GPT4_LP.csv')

MGB_row = range(0,6917)
VA_row = range(6917,13538)
UP_row = range(13538,24682)

In [34]:
common_pair, no_com_pair = get_final_pairs(sppmi_emb, name_desc, MGB_row, VA_row, UP_row, type_all = ['CCS','PheCode','RXNORM','LOINC'])

CCS&CCS
128 common pairs and 258 uncommon pairs.
CCS&PheCode
1735 common pairs and 4574 uncommon pairs.
CCS&RXNORM
345 common pairs and 3876 uncommon pairs.
CCS&LOINC
77 common pairs and 1091 uncommon pairs.
PheCode&PheCode
6586 common pairs and 17749 uncommon pairs.
PheCode&RXNORM
3029 common pairs and 28579 uncommon pairs.
PheCode&LOINC
602 common pairs and 8239 uncommon pairs.
RXNORM&RXNORM
681 common pairs and 7487 uncommon pairs.
RXNORM&LOINC
165 common pairs and 4595 uncommon pairs.
LOINC&LOINC
60 common pairs and 470 uncommon pairs.


In [35]:
len(common_pair)

13408

In [36]:
len(no_com_pair)

76918

In [39]:
common_df = get_dataframe(common_pair, name_desc)

In [40]:
no_com_df = get_dataframe(no_com_pair, name_desc)

In [41]:
common_df.to_csv('common_select_1109.csv', index=None)

In [42]:
no_com_df.to_csv('no_com_select_1109.csv', index=None)